In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


## use biopython read fasta

In [ ]:
from Bio import SeqIO

fasta_file = our_data + "Digitalis_leaf_PF00067_Merge98_Sort_Express.pep"

In [ ]:
sequences = []
for record in SeqIO.parse(fasta_file, "fasta"):
    sequences.append([record.id, str(record.seq)])

with open(
    our_data + "Digitalis_leaf_PF00067_Merge98_Sort_Express.fasta", "w"
) as output_handle:
    for seq_id, seq_data in sequences:
        output_handle.write(f">{seq_id}\n{seq_data}\n")

df_enzymedata = pd.DataFrame(sequences, columns=["enzyme", "Sequence"])

In [ ]:
df_enzymedata.to_csv(
    our_data + "Digitalis_leaf_PF00067_Merge98_Sort_Express.csv", index=False
)
df_enzymedata

,enzyme,Sequence
0,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...
1,Dcom_195433,MRANTAEKWITERAKRYGPVSKLSLFGNATVFIYGQSANKFLFADN...
2,Dcom_249831,MTLKFGMINIVVASSTEMAKEILQKKDQTFLGRPVPDAIRGEKGYE...
3,Dcom_272966,MGLPFFGEMLAFLWYFKVIRRPDDYINVKRRKYGDGEGLYRTHLFG...
4,Dcom_061020,MTHVTLSNLAKIHGPVMSFKLGTQLLIVGSSSAAAEEILKTNDRIL...
...,...,...
290,Droo_046339,MAHAVSSCVSYSHLLVNRQNELPSSRSPSSPVTHFSNCKRKGFSVR...
291,Dcom_143281,MASASSWLHKSFHDSLLCFFILSLTIMFLILFIFHLLRVRIWCNCE...
292,Droo_148538,MPNYQFMALISHFTTTPPPPTHDPFMSYSYFSNYLHSIIMASPNHS...
293,Droo_006827,MASSLSLLQLSPQILQTRGQRNRLSNVKSNGTGRDARRFRCSSSLN...


In [ ]:
data = {
    "substrate": [
        "3β-hydroxy-5β-pregnane-20-one",
        "5β-pregnane-3β,14β-diol-20-one",
        "digitoxigenin",
    ],
    "inchi": [
        "InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14-12-15(23)8-10-20(14,2)19(16)9-11-21(17,18)3/h14-19,23H,4-12H2,1-3H3/t14-,15+,16+,17-,18+,19+,20+,21-/m1/s1",
        "InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-4-14-12-15(23)6-9-19(14,2)17(18)7-10-20(16,21)3/h14-18,23-24H,4-12H2,1-3H3/t14-,15+,16-,17+,18-,19+,20-,21+/m1/s1",
        "InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4-19-18(21)6-9-22(2)17(7-10-23(19,22)26)14-11-20(25)27-13-14/h11,15-19,24,26H,3-10,12-13H2,1-2H3/t15-,16+,17-,18+,19-,21+,22-,23+/m1/s1",
    ],
}


df_substratedata = pd.DataFrame(data)
df_substratedata

,substrate,inchi
0,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...
1,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...
2,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...


In [ ]:
df_enzymedata["key"] = 1
df_substratedata["key"] = 1


result = pd.merge(df_enzymedata, df_substratedata, on="key", how="outer")
result.drop(columns="key", inplace=True)

In [7]:
result

,enzyme,Sequence,substrate,inchi
0,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...
1,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...
2,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...
3,Dcom_195433,MRANTAEKWITERAKRYGPVSKLSLFGNATVFIYGQSANKFLFADN...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...
4,Dcom_195433,MRANTAEKWITERAKRYGPVSKLSLFGNATVFIYGQSANKFLFADN...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...
...,...,...,...,...
880,Droo_006827,MASSLSLLQLSPQILQTRGQRNRLSNVKSNGTGRDARRFRCSSSLN...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...
881,Droo_006827,MASSLSLLQLSPQILQTRGQRNRLSNVKSNGTGRDARRFRCSSSLN...,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...
882,Dcom_165338,MSWWWAGAIGAAKKKLEEDDAPPKHSSVALIVGVTGIIGNSLAEIL...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...
883,Dcom_165338,MSWWWAGAIGAAKKKLEEDDAPPKHSSVALIVGVTGIIGNSLAEIL...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...


In [8]:
# ! python extract.py esm1b_t33_650M_UR50S {our_data}'Digitalis_leaf_PF00067_Merge98_Sort_Express.fasta' {our_data}esm --repr_layers 33 --include mean

In [ ]:
result["ESM1b"] = ""
result["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "esm/")

for ind in result.index:
    esms = torch.load(rep_dict + str(result["enzyme"][ind]) + ".pt")
    ecfps = Chem.MolFromInchi(result["inchi"][ind])
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(
        ecfps, radius=2, nBits=1024
    ).ToBitString()
    result["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    result["ECFP"][ind] = ecfpso
result["Binding"] = 0

In [10]:
result

,enzyme,Sequence,substrate,inchi,ESM1b,ECFP,Binding
0,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...,"[-0.06909024715423584, 0.1400468945503235, -0....",0000000000000000000000000010000001001000000000...,0
1,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[-0.06909024715423584, 0.1400468945503235, -0....",0000000000000000000010000010000001001000000000...,0
2,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...,"[-0.06909024715423584, 0.1400468945503235, -0....",0000000000000000000010000010000001001000000000...,0
3,Dcom_195433,MRANTAEKWITERAKRYGPVSKLSLFGNATVFIYGQSANKFLFADN...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...,"[-0.09775352478027344, 0.246835395693779, 0.03...",0000000000000000000000000010000001001000000000...,0
4,Dcom_195433,MRANTAEKWITERAKRYGPVSKLSLFGNATVFIYGQSANKFLFADN...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[-0.09775352478027344, 0.246835395693779, 0.03...",0000000000000000000010000010000001001000000000...,0
...,...,...,...,...,...,...,...
880,Droo_006827,MASSLSLLQLSPQILQTRGQRNRLSNVKSNGTGRDARRFRCSSSLN...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[-0.15221983194351196, 0.13631105422973633, 0....",0000000000000000000010000010000001001000000000...,0
881,Droo_006827,MASSLSLLQLSPQILQTRGQRNRLSNVKSNGTGRDARRFRCSSSLN...,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...,"[-0.15221983194351196, 0.13631105422973633, 0....",0000000000000000000010000010000001001000000000...,0
882,Dcom_165338,MSWWWAGAIGAAKKKLEEDDAPPKHSSVALIVGVTGIIGNSLAEIL...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...,"[0.0396469421684742, 0.2563888132572174, -0.05...",0000000000000000000000000010000001001000000000...,0
883,Dcom_165338,MSWWWAGAIGAAKKKLEEDDAPPKHSSVALIVGVTGIIGNSLAEIL...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[0.0396469421684742, 0.2563888132572174, -0.05...",0000000000000000000010000010000001001000000000...,0


In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

data_X, data_y = create_input_and_output_data(df=result)

In [ ]:
bst = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "p450normalmodel.dat")
)
dwant = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)
y_test_pred = bst.predict(dwant)
result["scores"] = y_test_pred

In [13]:
result

,enzyme,Sequence,substrate,inchi,ESM1b,ECFP,Binding,scores
0,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...,"[-0.06909024715423584, 0.1400468945503235, -0....",0000000000000000000000000010000001001000000000...,0,0.097201
1,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[-0.06909024715423584, 0.1400468945503235, -0....",0000000000000000000010000010000001001000000000...,0,0.056704
2,Droo_175168,VLLALPVILILLYLKQKSSGKRRRQPPGPSGLPFIGNLHQFDGIYP...,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...,"[-0.06909024715423584, 0.1400468945503235, -0....",0000000000000000000010000010000001001000000000...,0,0.016321
3,Dcom_195433,MRANTAEKWITERAKRYGPVSKLSLFGNATVFIYGQSANKFLFADN...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...,"[-0.09775352478027344, 0.246835395693779, 0.03...",0000000000000000000000000010000001001000000000...,0,0.154007
4,Dcom_195433,MRANTAEKWITERAKRYGPVSKLSLFGNATVFIYGQSANKFLFADN...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[-0.09775352478027344, 0.246835395693779, 0.03...",0000000000000000000010000010000001001000000000...,0,0.297599
...,...,...,...,...,...,...,...,...
880,Droo_006827,MASSLSLLQLSPQILQTRGQRNRLSNVKSNGTGRDARRFRCSSSLN...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[-0.15221983194351196, 0.13631105422973633, 0....",0000000000000000000010000010000001001000000000...,0,0.065117
881,Droo_006827,MASSLSLLQLSPQILQTRGQRNRLSNVKSNGTGRDARRFRCSSSLN...,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...,"[-0.15221983194351196, 0.13631105422973633, 0....",0000000000000000000010000010000001001000000000...,0,0.039451
882,Dcom_165338,MSWWWAGAIGAAKKKLEEDDAPPKHSSVALIVGVTGIIGNSLAEIL...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...,"[0.0396469421684742, 0.2563888132572174, -0.05...",0000000000000000000000000010000001001000000000...,0,0.049951
883,Dcom_165338,MSWWWAGAIGAAKKKLEEDDAPPKHSSVALIVGVTGIIGNSLAEIL...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[0.0396469421684742, 0.2563888132572174, -0.05...",0000000000000000000010000010000001001000000000...,0,0.086291


In [ ]:
result.to_csv(our_data + "onetest.csv")

In [ ]:
pivot_df = result.pivot(index="enzyme", columns="substrate", values="scores")

In [ ]:
pivot_df.to_csv(our_data + "onetest_3.csv")

In [ ]:
result[result["enzyme"] == "Dcom_266069"]

,enzyme,Sequence,substrate,inchi,ESM1b,ECFP,Binding,scores
483,Dcom_266069,MFYTIVALICAAFAIVYTWKILNFAWFKPKKLEKFLREQGFVGNSY...,3β-hydroxy-5β-pregnane-20-one,InChI=1S/C21H34O2/c1-13(22)17-6-7-18-16-5-4-14...,"[-0.07239473611116409, 0.15098997950553894, -0...",0000000000000000000000000010000001001000000000...,0,0.088752
484,Dcom_266069,MFYTIVALICAAFAIVYTWKILNFAWFKPKKLEKFLREQGFVGNSY...,"5β-pregnane-3β,14β-diol-20-one",InChI=1S/C21H34O3/c1-13(22)16-8-11-21(24)18-5-...,"[-0.07239473611116409, 0.15098997950553894, -0...",0000000000000000000010000010000001001000000000...,0,0.153346
485,Dcom_266069,MFYTIVALICAAFAIVYTWKILNFAWFKPKKLEKFLREQGFVGNSY...,digitoxigenin,InChI=1S/C23H34O4/c1-21-8-5-16(24)12-15(21)3-4...,"[-0.07239473611116409, 0.15098997950553894, -0...",0000000000000000000010000010000001001000000000...,0,0.154454
